In [10]:
"""
Gene-to-m/z Spatial Pattern Matching Pipeline V4
================================================

FIXES FROM V3:
1. Fixed artifact detection - was too aggressive (flagged everything)
2. Fixed loss function - contrastive was going negative
3. Fixed coherence calculation - was not discriminating
4. Removed over-filtering of m/z features
5. Better importance scoring based on feature variance and gradients
6. Proper visualization of all outputs

Author: V4 with critical fixes
"""

import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from torch_geometric.utils import softmax, add_self_loops
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.neighbors import NearestNeighbors
from scipy.spatial import distance_matrix
from scipy.stats import pearsonr, spearmanr
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import seaborn as sns
from typing import List, Dict, Tuple, Optional
import pandas as pd
import os
import warnings
from collections import defaultdict
from dataclasses import dataclass

warnings.filterwarnings('ignore')


# =============================================================================
# DATA CONFIGURATION
# =============================================================================

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_top_800/"
RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

AAD_TARGET_GENES = [
     'Thy1', 'App', 'Mapt','Apoe'
]


In [11]:


# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class SpatialSignature:
    """Spatial signature for a gene or m/z feature"""
    sample_id: str
    feature_name: str
    feature_type: str
    global_embedding: np.ndarray
    node_embeddings: np.ndarray
    node_importance: np.ndarray
    coordinates: np.ndarray
    raw_values: np.ndarray
    local_variance: np.ndarray  # High = interesting region
    gradient_magnitude: np.ndarray  # High = boundary/edge



In [12]:

# =============================================================================
# FIXED: SPATIAL FEATURE EXTRACTION (No over-aggressive filtering)
# =============================================================================

def compute_local_variance(
    coords: np.ndarray,
    values: np.ndarray,
    n_neighbors: int = 8
) -> np.ndarray:
    """
    Compute local variance - regions with HIGH variance are INTERESTING
    (boundaries, heterogeneous regions), not artifacts.
    """
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1)
    nn.fit(coords)
    _, indices = nn.kneighbors(coords)
    
    local_var = np.zeros(len(coords))
    
    for i in range(len(coords)):
        neighbor_vals = values[indices[i, 1:]]
        local_var[i] = np.var(neighbor_vals)
    
    # Normalize to [0, 1]
    if local_var.max() > local_var.min():
        local_var = (local_var - local_var.min()) / (local_var.max() - local_var.min())
    
    return local_var


def compute_gradient_magnitude(
    coords: np.ndarray,
    values: np.ndarray,
    n_neighbors: int = 8
) -> np.ndarray:
    """
    Compute spatial gradient magnitude - high at boundaries.
    """
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1)
    nn.fit(coords)
    distances, indices = nn.kneighbors(coords)
    
    gradient_mag = np.zeros(len(coords))
    
    for i in range(len(coords)):
        center_val = values[i]
        neighbor_vals = values[indices[i, 1:]]
        neighbor_dists = distances[i, 1:]
        
        # Gradient = change in value / distance
        val_diff = np.abs(neighbor_vals - center_val)
        gradients = val_diff / (neighbor_dists + 1e-8)
        gradient_mag[i] = np.mean(gradients)
    
    # Normalize
    if gradient_mag.max() > gradient_mag.min():
        gradient_mag = (gradient_mag - gradient_mag.min()) / (gradient_mag.max() - gradient_mag.min())
    
    return gradient_mag


def detect_extreme_outliers(
    values: np.ndarray,
    threshold: float = 5.0
) -> np.ndarray:
    """
    Detect only EXTREME outliers (very conservative).
    Uses IQR method which is robust.
    """
    q1 = np.percentile(values, 25)
    q3 = np.percentile(values, 75)
    iqr = q3 - q1
    
    # Very conservative: only flag points > Q3 + 5*IQR
    upper_bound = q3 + threshold * iqr
    
    outliers = values > upper_bound
    
    return outliers


def compute_biological_importance(
    coords: np.ndarray,
    values: np.ndarray,
    n_neighbors: int = 8
) -> np.ndarray:
    """
    Compute importance based on biological relevance:
    - High local variance = interesting heterogeneous region
    - High gradient = boundary/edge region
    - Not isolated extreme values
    
    Returns importance score [0, 1] where 1 = very important.
    """
    # Local variance component
    local_var = compute_local_variance(coords, values, n_neighbors)
    
    # Gradient component
    gradient = compute_gradient_magnitude(coords, values, n_neighbors)
    
    # Value magnitude (normalized)
    val_norm = (values - values.min()) / (values.max() - values.min() + 1e-8)
    
    # Combine: regions with high variance AND high values are important
    # Boundaries (high gradient) are also important
    importance = 0.4 * local_var + 0.3 * gradient + 0.3 * val_norm
    
    # Penalize isolated extreme outliers (likely artifacts)
    outliers = detect_extreme_outliers(values, threshold=5.0)
    
    # Check if outliers are isolated
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1)
    nn.fit(coords)
    _, indices = nn.kneighbors(coords)
    
    for i in np.where(outliers)[0]:
        neighbor_vals = values[indices[i, 1:]]
        # If neighbors have much lower values, this is isolated
        if values[i] > 3 * np.mean(neighbor_vals):
            importance[i] *= 0.3  # Reduce but don't eliminate
    
    return importance



In [13]:

# =============================================================================
# SPATIAL GRAPH CONSTRUCTION
# =============================================================================

def build_spatial_graph(
    coords: np.ndarray,
    n_neighbors: int = 8,
    grid_type: str = 'cartesian'
) -> torch.Tensor:
    """
    Build spatial graph with appropriate neighbors.
    
    Args:
        coords: Spatial coordinates
        n_neighbors: 6 for hexagonal (RNA), 8 for Cartesian (MSI)
        grid_type: 'hexagonal' or 'cartesian'
    """
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1)
    nn.fit(coords)
    distances, indices = nn.kneighbors(coords)
    
    # Adaptive distance threshold based on median neighbor distance
    median_dist = np.median(distances[:, 1])
    if grid_type == 'hexagonal':
        max_dist = median_dist * 1.3
    else:
        # Cartesian: diagonal neighbors are sqrt(2) times further
        max_dist = median_dist * 1.5
    
    edge_list = []
    for i in range(len(coords)):
        for j_idx in range(1, n_neighbors + 1):
            j = indices[i, j_idx]
            if distances[i, j_idx] <= max_dist:
                edge_list.append([i, j])
    
    # Make bidirectional and remove duplicates
    edge_set = set()
    for e in edge_list:
        edge_set.add((e[0], e[1]))
        edge_set.add((e[1], e[0]))
    
    edge_list = list(edge_set)
    
    if len(edge_list) == 0:
        # Fallback: just use k-NN without distance filter
        edge_list = []
        for i in range(len(coords)):
            for j in indices[i, 1:n_neighbors+1]:
                edge_list.append([i, j])
                edge_list.append([j, i])
    
    return torch.tensor(edge_list, dtype=torch.long).t().contiguous()



In [14]:

# =============================================================================
# IMPROVED GAT MODEL
# =============================================================================

class SpatialGATConv(nn.Module):
    """GAT convolution with spatial awareness"""
    
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        heads: int = 4,
        concat: bool = True,
        dropout: float = 0.1
    ):
        super().__init__()
        
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.heads = heads
        self.concat = concat
        
        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.att = nn.Parameter(torch.Tensor(1, heads, 2 * out_channels))
        
        if concat:
            self.bias = nn.Parameter(torch.Tensor(heads * out_channels))
        else:
            self.bias = nn.Parameter(torch.Tensor(out_channels))
        
        self.dropout = nn.Dropout(dropout)
        self.leaky_relu = nn.LeakyReLU(0.2)
        
        self.stored_attention = None
        self._reset_parameters()
    
    def _reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att)
        nn.init.zeros_(self.bias)
    
    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        pos: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        
        N = x.size(0)
        edge_index, _ = add_self_loops(edge_index, num_nodes=N)
        
        x_transformed = self.lin(x).view(N, self.heads, self.out_channels)
        
        src, dst = edge_index[0], edge_index[1]
        
        # Concatenate source and target features for attention
        alpha = torch.cat([x_transformed[src], x_transformed[dst]], dim=-1)
        alpha = (alpha * self.att).sum(dim=-1)
        alpha = self.leaky_relu(alpha)
        
        # Spatial distance weighting (optional, if pos provided)
        if pos is not None:
            dist = torch.norm(pos[src] - pos[dst], dim=-1, keepdim=True)
            dist_weight = torch.exp(-dist / (dist.mean() + 1e-8))
            alpha = alpha + 0.1 * dist_weight.expand(-1, self.heads)  # [E, heads] + [E, heads] -> works
        
        alpha = softmax(alpha, dst, num_nodes=N)
        self.stored_attention = alpha.detach()
        
        alpha = self.dropout(alpha)
        
        # Aggregate
        out = torch.zeros(N, self.heads, self.out_channels, device=x.device)
        alpha_expanded = alpha.unsqueeze(-1)
        weighted = alpha_expanded * x_transformed[src]
        out.scatter_add_(0, dst.view(-1, 1, 1).expand(-1, self.heads, self.out_channels), weighted)
        
        if self.concat:
            out = out.view(N, self.heads * self.out_channels)
        else:
            out = out.mean(dim=1)
        
        return out + self.bias


class SpatialGNN(nn.Module):
    """
    Spatial GNN for learning spatial patterns.
    """
    
    def __init__(
        self,
        input_dim: int = None,
        hidden_dim: int = 128,
        embedding_dim: int = 64,
        num_layers: int = 3,
        num_heads: int = 4,
        dropout: float = 0.1,
        projection_dim: int = 64
    ):
        super().__init__()
        
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim
        self.projection_dim = projection_dim
        
        # Input projections for different dimensions
        self.input_projections = nn.ModuleDict()
        if input_dim is not None:
            self.input_projections[str(input_dim)] = nn.Sequential(
                nn.Linear(input_dim, projection_dim),
                nn.LayerNorm(projection_dim),
                nn.GELU()
            )
        
        # GAT layers
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        
        self.convs.append(SpatialGATConv(
            projection_dim, hidden_dim, heads=num_heads, concat=True, dropout=dropout
        ))
        self.norms.append(nn.LayerNorm(hidden_dim * num_heads))
        
        for _ in range(num_layers - 2):
            self.convs.append(SpatialGATConv(
                hidden_dim * num_heads, hidden_dim, heads=num_heads,
                concat=True, dropout=dropout
            ))
            self.norms.append(nn.LayerNorm(hidden_dim * num_heads))
        
        self.convs.append(SpatialGATConv(
            hidden_dim * num_heads, embedding_dim, heads=1,
            concat=False, dropout=dropout
        ))
        self.norms.append(nn.LayerNorm(embedding_dim))
        
        # Importance prediction from embeddings
        self.importance_head = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
        # Global pooling attention
        self.pool_attention = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # Decoder for reconstruction
        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, projection_dim)
        )
        
        self.dropout = nn.Dropout(dropout)
    
    def _get_projection(self, input_dim: int) -> nn.Module:
        key = str(input_dim)
        if key not in self.input_projections:
            device = next(self.parameters()).device
            self.input_projections[key] = nn.Sequential(
                nn.Linear(input_dim, self.projection_dim),
                nn.LayerNorm(self.projection_dim),
                nn.GELU()
            ).to(device)
        return self.input_projections[key]
    
    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        pos: torch.Tensor,
        bio_importance: Optional[torch.Tensor] = None
    ) -> dict:
        """
        Forward pass.
        
        Args:
            x: Node features
            edge_index: Graph edges
            pos: Spatial positions
            bio_importance: Pre-computed biological importance (optional)
        """
        # Project input
        h_input = self._get_projection(x.size(1))(x)
        
        attention_weights = []
        h = h_input
        
        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            h_new = conv(h, edge_index, pos)
            attention_weights.append(conv.stored_attention)
            h_new = norm(h_new)
            if i < len(self.convs) - 1:
                h_new = F.gelu(h_new)
                h_new = self.dropout(h_new)
            h = h_new
        
        node_embeddings = h
        
        # Learned importance
        learned_importance = self.importance_head(node_embeddings).squeeze(-1)
        
        # Combine with biological importance if provided
        if bio_importance is not None:
            # Weight: 50% learned, 50% biological
            node_importance = 0.5 * learned_importance + 0.5 * bio_importance
        else:
            node_importance = learned_importance
        
        # Normalize importance
        node_importance = node_importance / (node_importance.max() + 1e-8)
        
        # Weighted embeddings
        weight = node_importance / (node_importance.sum() + 1e-8) * x.size(0)
        weighted_embeddings = node_embeddings * weight.unsqueeze(-1)
        
        # Global embedding via attention pooling
        attn_scores = self.pool_attention(node_embeddings).squeeze(-1)
        attn_weights = F.softmax(attn_scores, dim=0)
        global_embedding = (attn_weights.unsqueeze(-1) * weighted_embeddings).sum(dim=0)
        
        # Reconstruction
        reconstructed = self.decoder(node_embeddings)
        
        return {
            'node_embeddings': node_embeddings,
            'weighted_embeddings': weighted_embeddings,
            'global_embedding': global_embedding,
            'node_importance': node_importance,
            'learned_importance': learned_importance,
            'attention_weights': attention_weights,
            'reconstructed': reconstructed,
            'h_input': h_input
        }



In [15]:

# =============================================================================
# FIXED LOSS FUNCTION (No negative values)
# =============================================================================

class SpatialLoss(nn.Module):
    """
    Loss function that stays positive and meaningful.
    """
    
    def __init__(
        self,
        reconstruction_weight: float = 1.0,
        smoothness_weight: float = 0.3,
        structure_weight: float = 0.3
    ):
        super().__init__()
        self.reconstruction_weight = reconstruction_weight
        self.smoothness_weight = smoothness_weight
        self.structure_weight = structure_weight
    
    def forward(
        self,
        output: dict,
        pos: torch.Tensor,
        edge_index: torch.Tensor
    ) -> Tuple[torch.Tensor, dict]:
        
        losses = {}
        
        # 1. Reconstruction loss
        recon_loss = F.mse_loss(output['reconstructed'], output['h_input'])
        losses['reconstruction'] = recon_loss * self.reconstruction_weight
        
        # 2. Spatial smoothness - nearby nodes should have similar embeddings
        src, dst = edge_index[0], edge_index[1]
        emb_diff = output['node_embeddings'][src] - output['node_embeddings'][dst]
        smooth_loss = (emb_diff ** 2).sum(dim=-1).mean()
        # Normalize by embedding dimension
        smooth_loss = smooth_loss / output['node_embeddings'].size(-1)
        losses['smoothness'] = smooth_loss * self.smoothness_weight
        
        # 3. Structure preservation - embedding distance ~ spatial distance
        N = pos.size(0)
        n_samples = min(1000, N * (N - 1) // 2)
        
        idx1 = torch.randint(0, N, (n_samples,), device=pos.device)
        idx2 = torch.randint(0, N, (n_samples,), device=pos.device)
        mask = idx1 != idx2
        idx1, idx2 = idx1[mask], idx2[mask]
        
        if len(idx1) > 10:
            spatial_dist = torch.norm(pos[idx1] - pos[idx2], dim=-1)
            emb_dist = torch.norm(
                output['node_embeddings'][idx1] - output['node_embeddings'][idx2], 
                dim=-1
            )
            
            # Normalize both to [0, 1]
            spatial_dist = spatial_dist / (spatial_dist.max() + 1e-8)
            emb_dist = emb_dist / (emb_dist.max() + 1e-8)
            
            structure_loss = F.mse_loss(emb_dist, spatial_dist)
            losses['structure'] = structure_loss * self.structure_weight
        else:
            losses['structure'] = torch.tensor(0.0, device=pos.device)
        
        total = sum(losses.values())
        loss_dict = {k: v.item() for k, v in losses.items()}
        loss_dict['total'] = total.item()
        
        return total, loss_dict


In [16]:


# =============================================================================
# MAIN MATCHER CLASS
# =============================================================================

class GeneToMzMatcher:
    """
    Gene-to-m/z matcher with proper importance scoring.
    """
    
    def __init__(
        self,
        output_dir: str = './gene_to_mz_results_v4',
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        self.device = device
        self.output_dir = output_dir
        
        # Create all output directories
        subdirs = [
            'saliency_maps', 'gene_visualizations', 'training_curves',
            'embeddings', 'match_visualizations'
        ]
        for subdir in subdirs:
            os.makedirs(os.path.join(output_dir, subdir), exist_ok=True)
        
        self.rna_data = {}
        self.msi_data = {}
        self.rna_model = None
        self.msi_model = None
        
        self.gene_signatures = defaultdict(dict)
        self.mz_signatures = defaultdict(dict)
    
    def load_all_data(self):
        """Load all data"""
        print("Loading RNA-seq data...")
        for file, sample_id in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
            path = os.path.join(RNA_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.rna_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.rna_data[sample_id].shape}")
        
        print("\nLoading MSI data...")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
    
    def check_gene_availability(self) -> Dict[str, Dict[str, bool]]:
        """Check gene availability"""
        gene_availability = {}
        for gene in AAD_TARGET_GENES:
            gene_availability[gene] = {}
            for sample_id in RNA_SAMPLE_IDS:
                if sample_id in self.rna_data:
                    gene_availability[gene][sample_id] = gene in self.rna_data[sample_id].var_names
                else:
                    gene_availability[gene][sample_id] = False
        
        print("\nGene availability:")
        for gene in AAD_TARGET_GENES:
            available = sum(gene_availability[gene].values())
            print(f"  {gene}: {available}/{len(RNA_SAMPLE_IDS)} samples")
        
        return gene_availability
    
    def prepare_data(
        self,
        coords: np.ndarray,
        features: np.ndarray,
        n_neighbors: int,
        grid_type: str
    ) -> Tuple[Data, np.ndarray]:
        """
        Prepare graph data with biological importance.
        """
        if features.ndim == 1:
            features = features.reshape(-1, 1)
        
        # Compute biological importance for each feature column
        bio_importance = np.zeros(len(coords))
        for col in range(features.shape[1]):
            col_importance = compute_biological_importance(coords, features[:, col], n_neighbors)
            bio_importance += col_importance
        bio_importance /= features.shape[1]
        
        # Scale features
        scaler = RobustScaler()
        features_scaled = scaler.fit_transform(features)
        
        # Build graph
        edge_index = build_spatial_graph(coords, n_neighbors, grid_type)
        
        # Normalize positions
        pos = (coords - coords.mean(axis=0)) / (coords.std(axis=0) + 1e-8)
        
        data = Data(
            x=torch.tensor(features_scaled, dtype=torch.float32),
            edge_index=edge_index,
            pos=torch.tensor(pos, dtype=torch.float32)
        )
        
        return data, bio_importance
    
    def train_model(
        self,
        data_dict: Dict[str, Data],
        bio_importance_dict: Dict[str, np.ndarray],
        model_type: str,
        epochs: int = 200,
        lr: float = 0.001
    ) -> SpatialGNN:
        """Train spatial GNN model"""
        print(f"\nTraining {model_type.upper()} model...")
        
        first_data = list(data_dict.values())[0]
        input_dim = first_data.x.size(1)
        
        model = SpatialGNN(
            input_dim=input_dim,
            hidden_dim=128,
            embedding_dim=64,
            num_layers=3,
            num_heads=4,
            projection_dim=64
        ).to(self.device)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        
        # Cosine annealing with warmup
        warmup_epochs = 20
        def lr_lambda(epoch):
            if epoch < warmup_epochs:
                return (epoch + 1) / warmup_epochs
            progress = (epoch - warmup_epochs) / (epochs - warmup_epochs)
            return 0.5 * (1 + np.cos(np.pi * progress))
        
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        loss_fn = SpatialLoss()
        
        model.train()
        loss_history = []
        
        for epoch in range(epochs):
            total_loss = 0
            
            for sample_id, data in data_dict.items():
                data = data.to(self.device)
                bio_imp = torch.tensor(
                    bio_importance_dict[sample_id], 
                    dtype=torch.float32, 
                    device=self.device
                )
                
                optimizer.zero_grad()
                
                output = model(data.x, data.edge_index, data.pos, bio_imp)
                loss, _ = loss_fn(output, data.pos, data.edge_index)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                total_loss += loss.item()
            
            scheduler.step()
            avg_loss = total_loss / len(data_dict)
            loss_history.append(avg_loss)
            
            if (epoch + 1) % 50 == 0:
                print(f"  Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
        
        print(f"  Final loss: {loss_history[-1]:.4f}")
        
        # Plot loss curve
        plt.figure(figsize=(10, 5))
        plt.plot(loss_history)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_type.upper()} Training Loss')
        plt.grid(True, alpha=0.3)
        plt.savefig(
            os.path.join(self.output_dir, 'training_curves', f'{model_type}_loss.png'),
            dpi=150, bbox_inches='tight'
        )
        plt.close()
        
        return model
    
    def extract_gene_signature(
        self,
        sample_id: str,
        gene: str,
        model: SpatialGNN
    ) -> Optional[SpatialSignature]:
        """Extract spatial signature for a gene"""
        if sample_id not in self.rna_data:
            return None
        
        adata = self.rna_data[sample_id]
        if gene not in adata.var_names:
            return None
        
        coords = np.column_stack([
            adata.obs['x_um'].values,
            adata.obs['y_um'].values
        ])
        
        gene_idx = list(adata.var_names).index(gene)
        if hasattr(adata.X, 'toarray'):
            gene_expr = adata.X[:, gene_idx].toarray().flatten()
        else:
            gene_expr = adata.X[:, gene_idx].flatten()
        
        # Compute biological importance
        bio_importance = compute_biological_importance(coords, gene_expr, n_neighbors=6)
        local_var = compute_local_variance(coords, gene_expr, n_neighbors=6)
        gradient = compute_gradient_magnitude(coords, gene_expr, n_neighbors=6)
        
        # Prepare data
        data, _ = self.prepare_data(coords, gene_expr, n_neighbors=6, grid_type='hexagonal')
        data = data.to(self.device)
        bio_imp_tensor = torch.tensor(bio_importance, dtype=torch.float32, device=self.device)
        
        # Extract embeddings
        model.eval()
        with torch.no_grad():
            output = model(data.x, data.edge_index, data.pos, bio_imp_tensor)
        
        sig = SpatialSignature(
            sample_id=sample_id,
            feature_name=gene,
            feature_type='gene',
            global_embedding=output['global_embedding'].cpu().numpy(),
            node_embeddings=output['weighted_embeddings'].cpu().numpy(),
            node_importance=output['node_importance'].cpu().numpy(),
            coordinates=coords,
            raw_values=gene_expr,
            local_variance=local_var,
            gradient_magnitude=gradient
        )
        
        self.gene_signatures[gene][sample_id] = sig
        return sig
    
    def extract_mz_signature(
        self,
        sample_id: str,
        mz_name: str,
        mz_values: np.ndarray,
        coords: np.ndarray,
        model: SpatialGNN
    ) -> SpatialSignature:
        """Extract spatial signature for an m/z feature"""
        
        # Compute biological importance
        bio_importance = compute_biological_importance(coords, mz_values, n_neighbors=8)
        local_var = compute_local_variance(coords, mz_values, n_neighbors=8)
        gradient = compute_gradient_magnitude(coords, mz_values, n_neighbors=8)
        
        # Prepare data
        data, _ = self.prepare_data(coords, mz_values, n_neighbors=8, grid_type='cartesian')
        data = data.to(self.device)
        bio_imp_tensor = torch.tensor(bio_importance, dtype=torch.float32, device=self.device)
        
        # Extract embeddings
        model.eval()
        with torch.no_grad():
            output = model(data.x, data.edge_index, data.pos, bio_imp_tensor)
        
        sig = SpatialSignature(
            sample_id=sample_id,
            feature_name=mz_name,
            feature_type='mz',
            global_embedding=output['global_embedding'].cpu().numpy(),
            node_embeddings=output['weighted_embeddings'].cpu().numpy(),
            node_importance=output['node_importance'].cpu().numpy(),
            coordinates=coords,
            raw_values=mz_values,
            local_variance=local_var,
            gradient_magnitude=gradient
        )
        
        return sig
    
    def compute_similarity(
        self,
        gene_sig: SpatialSignature,
        mz_sig: SpatialSignature
    ) -> dict:
        """Compute similarity between signatures"""
        
        # Embedding similarity
        g_emb = gene_sig.global_embedding.flatten()
        m_emb = mz_sig.global_embedding.flatten()
        
        cosine = np.dot(g_emb, m_emb) / (np.linalg.norm(g_emb) * np.linalg.norm(m_emb) + 1e-8)
        euclidean = 1 / (1 + np.linalg.norm(g_emb - m_emb))
        
        # Importance overlap
        imp_overlap = self._compute_importance_overlap(gene_sig, mz_sig)
        
        # Raw correlation
        raw_pearson = self._compute_raw_correlation(gene_sig, mz_sig)
        
        return {
            'embedding_cosine': cosine,
            'embedding_euclidean': euclidean,
            'importance_overlap': imp_overlap,
            'raw_pearson': raw_pearson
        }
    
    def _compute_importance_overlap(
        self,
        sig1: SpatialSignature,
        sig2: SpatialSignature,
        grid_res: int = 50
    ) -> float:
        """Compute importance overlap via grid interpolation"""
        x_min = min(sig1.coordinates[:, 0].min(), sig2.coordinates[:, 0].min())
        x_max = max(sig1.coordinates[:, 0].max(), sig2.coordinates[:, 0].max())
        y_min = min(sig1.coordinates[:, 1].min(), sig2.coordinates[:, 1].min())
        y_max = max(sig1.coordinates[:, 1].max(), sig2.coordinates[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_res),
            np.linspace(y_min, y_max, grid_res)
        )
        
        imp1 = griddata(sig1.coordinates, sig1.node_importance, (grid_x, grid_y),
                       method='linear', fill_value=0)
        imp2 = griddata(sig2.coordinates, sig2.node_importance, (grid_x, grid_y),
                       method='linear', fill_value=0)
        
        imp1 = imp1 / (imp1.max() + 1e-8)
        imp2 = imp2 / (imp2.max() + 1e-8)
        
        intersection = np.minimum(imp1, imp2).sum()
        union = np.maximum(imp1, imp2).sum()
        
        return intersection / (union + 1e-8)
    
    def _compute_raw_correlation(
        self,
        sig1: SpatialSignature,
        sig2: SpatialSignature,
        grid_res: int = 50
    ) -> float:
        """Compute raw value correlation"""
        x_min = min(sig1.coordinates[:, 0].min(), sig2.coordinates[:, 0].min())
        x_max = max(sig1.coordinates[:, 0].max(), sig2.coordinates[:, 0].max())
        y_min = min(sig1.coordinates[:, 1].min(), sig2.coordinates[:, 1].min())
        y_max = max(sig1.coordinates[:, 1].max(), sig2.coordinates[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_res),
            np.linspace(y_min, y_max, grid_res)
        )
        
        v1 = griddata(sig1.coordinates, sig1.raw_values, (grid_x, grid_y),
                     method='linear', fill_value=np.nan)
        v2 = griddata(sig2.coordinates, sig2.raw_values, (grid_x, grid_y),
                     method='linear', fill_value=np.nan)
        
        mask = ~(np.isnan(v1) | np.isnan(v2))
        if mask.sum() < 10:
            return 0.0
        
        r, _ = pearsonr(v1[mask], v2[mask])
        return r if not np.isnan(r) else 0.0
    
    def find_matches(
        self,
        gene_sig: SpatialSignature,
        mz_signatures: Dict[str, SpatialSignature],
        top_k: int = 20
    ) -> pd.DataFrame:
        """Find best matching m/z features"""
        matches = []
        
        for mz_name, mz_sig in mz_signatures.items():
            sim = self.compute_similarity(gene_sig, mz_sig)
            
            combined = (
                0.35 * sim['embedding_cosine'] +
                0.30 * sim['importance_overlap'] +
                0.20 * max(sim['raw_pearson'], 0) +
                0.15 * sim['embedding_euclidean']
            )
            
            matches.append({
                'gene': gene_sig.feature_name,
                'rna_sample': gene_sig.sample_id,
                'mz_feature': mz_name,
                'msi_sample': mz_sig.sample_id,
                **sim,
                'combined_score': combined
            })
        
        df = pd.DataFrame(matches)
        return df.sort_values('combined_score', ascending=False).head(top_k)
    
    def visualize_signature(
        self,
        sig: SpatialSignature,
        save_path: str
    ):
        """Visualize spatial signature with multiple views"""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # 1. Raw values
        im1 = axes[0, 0].scatter(
            sig.coordinates[:, 0], sig.coordinates[:, 1],
            c=sig.raw_values, cmap='viridis', s=10, alpha=0.8
        )
        axes[0, 0].set_title(f'Raw Values: {sig.feature_name}', fontweight='bold')
        axes[0, 0].set_xlabel('X (μm)')
        axes[0, 0].set_ylabel('Y (μm)')
        plt.colorbar(im1, ax=axes[0, 0])
        
        # 2. Local variance (high = interesting region)
        im2 = axes[0, 1].scatter(
            sig.coordinates[:, 0], sig.coordinates[:, 1],
            c=sig.local_variance, cmap='hot', s=10, alpha=0.8
        )
        axes[0, 1].set_title('Local Variance (High = Heterogeneous)', fontweight='bold')
        plt.colorbar(im2, ax=axes[0, 1])
        
        # 3. Gradient magnitude (high = boundary)
        im3 = axes[0, 2].scatter(
            sig.coordinates[:, 0], sig.coordinates[:, 1],
            c=sig.gradient_magnitude, cmap='plasma', s=10, alpha=0.8
        )
        axes[0, 2].set_title('Gradient (High = Boundary)', fontweight='bold')
        plt.colorbar(im3, ax=axes[0, 2])
        
        # 4. Learned importance
        im4 = axes[1, 0].scatter(
            sig.coordinates[:, 0], sig.coordinates[:, 1],
            c=sig.node_importance, cmap='YlOrRd', s=10, alpha=0.8
        )
        axes[1, 0].set_title('Combined Importance', fontweight='bold')
        plt.colorbar(im4, ax=axes[1, 0])
        
        # 5. Importance-weighted values
        weighted = sig.raw_values * sig.node_importance
        weighted = (weighted - weighted.min()) / (weighted.max() - weighted.min() + 1e-8)
        im5 = axes[1, 1].scatter(
            sig.coordinates[:, 0], sig.coordinates[:, 1],
            c=weighted, cmap='magma', s=10, alpha=0.8
        )
        axes[1, 1].set_title('Importance-Weighted Values', fontweight='bold')
        plt.colorbar(im5, ax=axes[1, 1])
        
        # 6. Top important regions
        threshold = np.percentile(sig.node_importance, 80)
        important_mask = sig.node_importance >= threshold
        
        axes[1, 2].scatter(
            sig.coordinates[:, 0], sig.coordinates[:, 1],
            c='lightgray', s=5, alpha=0.3
        )
        sc = axes[1, 2].scatter(
            sig.coordinates[important_mask, 0],
            sig.coordinates[important_mask, 1],
            c=sig.raw_values[important_mask], cmap='Reds', s=15, alpha=0.9
        )
        axes[1, 2].set_title(f'Top 20% Important ({important_mask.sum()} points)', fontweight='bold')
        plt.colorbar(sc, ax=axes[1, 2])
        
        plt.suptitle(
            f'{sig.feature_type.upper()}: {sig.feature_name} ({sig.sample_id})',
            fontsize=14, fontweight='bold'
        )
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def visualize_match(
        self,
        gene_sig: SpatialSignature,
        mz_sig: SpatialSignature,
        similarity: dict,
        save_path: str
    ):
        """Visualize a gene-m/z match"""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # Row 1: Gene
        im1 = axes[0, 0].scatter(
            gene_sig.coordinates[:, 0], gene_sig.coordinates[:, 1],
            c=gene_sig.raw_values, cmap='viridis', s=20, alpha=0.8
        )
        axes[0, 0].set_title(f'Gene: {gene_sig.feature_name}', fontweight='bold')
        plt.colorbar(im1, ax=axes[0, 0])
        
        im2 = axes[0, 1].scatter(
            gene_sig.coordinates[:, 0], gene_sig.coordinates[:, 1],
            c=gene_sig.node_importance, cmap='hot', s=20, alpha=0.8
        )
        axes[0, 1].set_title('Gene Importance', fontweight='bold')
        plt.colorbar(im2, ax=axes[0, 1])
        
        # Row 2: m/z
        im3 = axes[1, 0].scatter(
            mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
            c=mz_sig.raw_values, cmap='viridis', s=5, alpha=0.8
        )
        axes[1, 0].set_title(f'm/z: {mz_sig.feature_name}', fontweight='bold')
        plt.colorbar(im3, ax=axes[1, 0])
        
        im4 = axes[1, 1].scatter(
            mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
            c=mz_sig.node_importance, cmap='hot', s=5, alpha=0.8
        )
        axes[1, 1].set_title('m/z Importance', fontweight='bold')
        plt.colorbar(im4, ax=axes[1, 1])
        
        # Correlation plot
        grid_res = 50
        x_min = min(gene_sig.coordinates[:, 0].min(), mz_sig.coordinates[:, 0].min())
        x_max = max(gene_sig.coordinates[:, 0].max(), mz_sig.coordinates[:, 0].max())
        y_min = min(gene_sig.coordinates[:, 1].min(), mz_sig.coordinates[:, 1].min())
        y_max = max(gene_sig.coordinates[:, 1].max(), mz_sig.coordinates[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_res),
            np.linspace(y_min, y_max, grid_res)
        )
        
        v1 = griddata(gene_sig.coordinates, gene_sig.raw_values, (grid_x, grid_y), method='linear')
        v2 = griddata(mz_sig.coordinates, mz_sig.raw_values, (grid_x, grid_y), method='linear')
        
        mask = ~(np.isnan(v1) | np.isnan(v2))
        if mask.sum() > 0:
            axes[0, 2].scatter(v1[mask], v2[mask], alpha=0.3, s=10)
            axes[0, 2].set_xlabel('Gene Expression')
            axes[0, 2].set_ylabel('m/z Intensity')
            axes[0, 2].set_title(f'Spatial Correlation (r={similarity["raw_pearson"]:.3f})', fontweight='bold')
        
        # Metrics
        metrics_text = f"""
Similarity Metrics:
─────────────────────
Embedding Cosine:  {similarity['embedding_cosine']:.4f}
Embedding Euclid:  {similarity['embedding_euclidean']:.4f}
Importance Overlap: {similarity['importance_overlap']:.4f}
Raw Pearson:       {similarity['raw_pearson']:.4f}
─────────────────────
Combined Score:    {similarity.get('combined_score', 0):.4f}
        """
        axes[1, 2].text(0.1, 0.5, metrics_text, transform=axes[1, 2].transAxes,
                       fontsize=11, verticalalignment='center', family='monospace',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        axes[1, 2].axis('off')
        
        plt.suptitle(
            f'Match: {gene_sig.feature_name} ↔ m/z {mz_sig.feature_name}',
            fontsize=14, fontweight='bold'
        )
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def run_analysis(
        self,
        epochs: int = 200,
        top_k: int = 20,
        samples_per_group: int = 2
    ) -> pd.DataFrame:
        """Run complete analysis"""
        print("\n" + "="*70)
        print("GENE-TO-M/Z MATCHING V4")
        print("="*70)
        
        gene_availability = self.check_gene_availability()
        
        # ===== TRAINING =====
        print("\n" + "-"*70)
        print("PHASE 1: Training Models")
        print("-"*70)
        
        # Prepare RNA training data (6 neighbors, hexagonal)
        print("\nPreparing RNA data (6 neighbors, hexagonal)...")
        rna_train = {}
        rna_bio_imp = {}
        
        for sample_id in list(self.rna_data.keys())[:4]:
            adata = self.rna_data[sample_id]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            if hasattr(adata.X, 'toarray'):
                features = adata.X[:, :50].toarray()
            else:
                features = adata.X[:, :50].copy()
            
            data, bio_imp = self.prepare_data(coords, features, n_neighbors=6, grid_type='hexagonal')
            rna_train[sample_id] = data
            rna_bio_imp[sample_id] = bio_imp
            print(f"  {sample_id}: {data.x.shape}, edges: {data.edge_index.shape[1]}")
        
        self.rna_model = self.train_model(rna_train, rna_bio_imp, 'rna', epochs)
        
        # Prepare MSI training data (8 neighbors, Cartesian)
        print("\nPreparing MSI data (8 neighbors, Cartesian)...")
        msi_train = {}
        msi_bio_imp = {}
        
        for sample_id in list(self.msi_data.keys())[:4]:
            adata = self.msi_data[sample_id]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            if hasattr(adata.X, 'toarray'):
                features = adata.X[:, :50].toarray()
            else:
                features = adata.X[:, :50].copy()
            
            data, bio_imp = self.prepare_data(coords, features, n_neighbors=8, grid_type='cartesian')
            msi_train[sample_id] = data
            msi_bio_imp[sample_id] = bio_imp
            print(f"  {sample_id}: {data.x.shape}, edges: {data.edge_index.shape[1]}")
        
        self.msi_model = self.train_model(msi_train, msi_bio_imp, 'msi', epochs)
        
        # ===== MATCHING =====
        print("\n" + "-"*70)
        print("PHASE 2: Extracting Signatures and Matching")
        print("-"*70)
        
        all_results = []
        groups = ['YC', 'YAD', 'AC', 'AAD']
        
        for gene in AAD_TARGET_GENES:
            print(f"\n{'='*50}")
            print(f"Gene: {gene}")
            print(f"{'='*50}")
            
            available = [s for s, a in gene_availability[gene].items() if a]
            if not available:
                print("  Not available in any sample")
                continue
            
            # Sample from each group
            samples_to_process = []
            for group in groups:
                group_samples = [s for s in available if s.startswith(group)]
                samples_to_process.extend(group_samples[:samples_per_group])
            
            for rna_sample in samples_to_process:
                print(f"\n  Sample: {rna_sample}")
                
                # Extract gene signature
                gene_sig = self.extract_gene_signature(rna_sample, gene, self.rna_model)
                if gene_sig is None:
                    print("    Failed to extract gene signature")
                    continue
                
                # Save saliency visualization
                self.visualize_signature(
                    gene_sig,
                    os.path.join(self.output_dir, 'saliency_maps', f'{gene}_{rna_sample}.png')
                )
                
                # Find matching MSI sample
                msi_sample = rna_sample
                if msi_sample not in self.msi_data:
                    print(f"    MSI sample {msi_sample} not found")
                    continue
                
                print(f"    Matching vs MSI {msi_sample}...")
                
                adata = self.msi_data[msi_sample]
                coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                
                if hasattr(adata.X, 'toarray'):
                    msi_data = adata.X.toarray()
                else:
                    msi_data = adata.X.copy()
                
                mz_names = list(adata.var_names)
                
                # Extract ALL m/z signatures (no filtering)
                print(f"    Processing {len(mz_names)} m/z features...")
                mz_sigs = {}
                
                for i, mz_name in enumerate(mz_names):
                    mz_values = msi_data[:, i]
                    sig = self.extract_mz_signature(
                        msi_sample, mz_name, mz_values, coords, self.msi_model
                    )
                    mz_sigs[mz_name] = sig
                    
                    if (i + 1) % 100 == 0:
                        print(f"      {i+1}/{len(mz_names)}")
                
                # Find matches
                matches = self.find_matches(gene_sig, mz_sigs, top_k)
                all_results.append(matches)
                
                # Print and visualize top matches
                if len(matches) > 0:
                    print(f"\n    Top 5 matches:")
                    for idx in range(min(5, len(matches))):
                        m = matches.iloc[idx]
                        print(f"      {idx+1}. m/z {m['mz_feature']}: "
                              f"combined={m['combined_score']:.3f}, "
                              f"cosine={m['embedding_cosine']:.3f}, "
                              f"overlap={m['importance_overlap']:.3f}, "
                              f"pearson={m['raw_pearson']:.3f}")
                    
                    # Visualize top match
                    top_match = matches.iloc[0]
                    top_mz_sig = mz_sigs[top_match['mz_feature']]
                    
                    self.visualize_match(
                        gene_sig, top_mz_sig,
                        {k: top_match[k] for k in ['embedding_cosine', 'embedding_euclidean',
                                                    'importance_overlap', 'raw_pearson', 'combined_score']},
                        os.path.join(
                            self.output_dir, 'match_visualizations',
                            f'{gene}_{rna_sample}_top_match.png'
                        )
                    )
        
        # Combine and save results
        if all_results:
            results = pd.concat(all_results, ignore_index=True)
            results = results.sort_values('combined_score', ascending=False)
            
            output_file = os.path.join(self.output_dir, 'gene_to_mz_matches_v4.csv')
            results.to_csv(output_file, index=False)
            print(f"\n\nResults saved to: {output_file}")
            
            # Summary
            print("\n" + "="*70)
            print("TOP MATCHES PER GENE")
            print("="*70)
            
            for gene in AAD_TARGET_GENES:
                gene_results = results[results['gene'] == gene]
                if len(gene_results) > 0:
                    top = gene_results.iloc[0]
                    print(f"\n{gene}:")
                    print(f"  Best m/z: {top['mz_feature']}")
                    print(f"  Sample: {top['rna_sample']}")
                    print(f"  Combined: {top['combined_score']:.3f}")
                    print(f"  Cosine: {top['embedding_cosine']:.3f}")
                    print(f"  Overlap: {top['importance_overlap']:.3f}")
                    print(f"  Pearson: {top['raw_pearson']:.3f}")
            
            return results
        
        return None


In [17]:


# =============================================================================
# MAIN
# =============================================================================

def main():
    print("="*70)
    print("GENE-TO-M/Z MATCHING V4")
    print("Fixed: Loss function, importance scoring, no over-filtering")
    print("="*70)
    
    matcher = GeneToMzMatcher(output_dir='./gene_to_mz_results_v4')
    matcher.load_all_data()
    
    results = matcher.run_analysis(epochs=200, top_k=40, samples_per_group=2)
    
    print("\n" + "="*70)
    print("COMPLETE!")
    print("="*70)
    print(f"\nOutput directory: {matcher.output_dir}/")
    print("  - saliency_maps/: Gene spatial importance visualizations")
    print("  - match_visualizations/: Top gene-m/z match visualizations")
    print("  - training_curves/: Loss curves")
    print("  - gene_to_mz_matches_v4.csv: All matches")
    
    return matcher, results


In [18]:


if __name__ == "__main__":
    matcher, results = main()

GENE-TO-M/Z MATCHING V4
Fixed: Loss function, importance scoring, no over-filtering
Loading RNA-seq data...
  YC_1: (2112, 800)
  YC_2: (2775, 800)
  YC_3: (2808, 800)
  YC_4: (2725, 800)
  YAD_1: (2915, 800)
  YAD_2: (2960, 800)
  YAD_3: (2880, 800)
  YAD_4: (2939, 800)
  AC_1: (3065, 800)
  AC_2: (3054, 800)
  AC_3: (2892, 800)
  AC_4: (3002, 800)
  AAD_1: (2700, 800)
  AAD_2: (2171, 800)
  AAD_3: (1584, 800)
  AAD_4: (2438, 800)

Loading MSI data...
  YC_1: (6688, 528)
  YC_2: (7858, 528)
  YC_3: (7150, 528)
  YC_4: (6067, 528)
  YAD_1: (7517, 528)
  YAD_2: (7596, 528)
  YAD_3: (6844, 528)
  YAD_4: (7591, 528)
  AC_1: (6955, 528)
  AC_2: (5729, 528)
  AC_3: (7569, 528)
  AC_4: (7792, 528)
  AAD_1: (6471, 528)
  AAD_2: (5959, 528)
  AAD_3: (5392, 528)
  AAD_4: (6833, 528)

GENE-TO-M/Z MATCHING V4

Gene availability:
  Thy1: 16/16 samples
  App: 16/16 samples
  Mapt: 16/16 samples
  Apoe: 16/16 samples

----------------------------------------------------------------------
PHASE 1: Tr